# 最近 12 市场只读复盘
从项目目录或本文件目录运行；只读取本地证据，生成同目录复盘材料。完整公式与脚本源码在后续单元内。

In [1]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/polymaker').is_dir())
import os, sys
os.chdir(ROOT)
REVIEW = ROOT / 'docs/development/reviews/20261017_2245_2345'
sys.path.insert(0, str(REVIEW))
"""Read-only reconstruction of 12 closed BTC markets; run from repository root."""
import csv
import hashlib
import json
import math
import statistics
from collections import Counter, defaultdict
from datetime import datetime, timedelta, timezone
from decimal import Decimal
from pathlib import Path

OUT = REVIEW
TZ = timezone(timedelta(hours=8))
START, END = 1792248300, 1792251900
D = lambda x: Decimal(str(x))


def stamp(t):
    return datetime.fromtimestamp(float(t), TZ).strftime("%H:%M:%S.%f")[:-3] if t is not None else "—"


def percentile(values, p):
    return sorted(values)[max(0, math.ceil(len(values)*p)-1)] if values else None


def write_csv(name, rows):
    if rows:
        with (OUT / name).open("w") as file:
            writer = csv.DictWriter(file, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)


def analyze():
    markets, windows, fills_out, events, provenance = [], [], [], [], []
    errors, paused, requests, final_outcomes, waits_all = [], [], [], Counter(), Counter()
    signal_histories, early_signals = [], []
    for path in Path("journal").glob("rdp_sm_*.jsonl"):
        with path.open() as file:
            first_line = next(file, "")
        if not first_line.strip():
            continue
        try:
            head = json.loads(first_line)
        except ValueError:
            continue
        if head["kind"] != "sm_session":
            continue
        session = head["data"]["payload"]
        epoch = int(session["slug"].rsplit("-", 1)[-1])
        if not START <= epoch < END:
            continue
        raw = path.read_bytes()
        rows = []
        for line, value in enumerate(raw.decode().splitlines(), 1):
            try:
                x = json.loads(value)
                rows.append((line, x, x["data"]["payload"]))
            except (ValueError, KeyError) as exc:
                errors.append([str(path), line, type(exc).__name__])
        by = defaultdict(list)
        for line, x, q in rows:
            by[x["kind"]].append((line, x["ts"], q))
        assert len(by["sm_run_result"]) == 1, path
        result_line, stopped, result = by["sm_run_result"][0]
        sid = session["session_id"]
        label = stamp(epoch)[:5]
        net = by["sm_network_request"]
        offset = statistics.median(q["completed_wall"]-q["completed_monotonic"] for _, _, q in net)
        source = str(path.resolve())
        provenance.append(dict(market=label, session_id=sid, slug=session["slug"],
                               path=source, sha256=hashlib.sha256(raw).hexdigest(),
                               bytes=len(raw), lines=len(rows), result_line=result_line))
        intents = {q["intent"]["window_id"]: (line, q["intent"], q.get("sizing", {}))
                   for line, _, q in by["sm_obi_final_decision"]}
        closes = {q["window_id"]: (line, q) for line, _, q in by["sm_obi_close_requested"]}
        results = {q["window_id"]: q for _, _, q in by["sm_window_result"]}
        placements = {q["quote"]["window_id"]: q for _, _, q in by["sm_placement_result"]}
        quote_by_oid, token_side = {}, {}
        for _, intent, _ in intents.values():
            for side in ("yes", "no"):
                token_side[intent["snapshot"][side]["token_id"]] = side.upper()
            w = intent["window_id"]
            q = placements[w]
            quote_by_oid[q.get("order_id") or q["expected_order_id"]] = intent["quotes"][0]
        official_times = {}
        for _, _, q in by["sm_official_reconciliation"]:
            for trade in q.get("trades", []):
                for leg in trade.get("maker_orders", []):
                    if leg.get("order_id") in quote_by_oid and trade.get("match_time"):
                        official_times[(trade["id"], leg["order_id"])] = float(trade["match_time"])
        confirmed = {}
        observations = defaultdict(dict)
        for line, t, fill in by["sm_trade"]:
            key = (fill["trade_id"], fill["maker_order_id"])
            observations[key].setdefault(fill["status"], t)
            if fill["status"] == "CONFIRMED":
                if key in confirmed:
                    assert confirmed[key][2] == fill, (path, key, "conflicting fill")
                confirmed[key] = (line, t, fill)
        costs, quantities = defaultdict(Decimal), defaultdict(Decimal)
        order_fills = defaultdict(list)
        for key, (line, t, fill) in confirmed.items():
            side = token_side[fill["token_id"]]
            quantities[side] += D(fill["size"])
            costs[side] += D(fill["size"]) * D(fill["price"])
            actual = official_times.get(key)
            record = dict(market=label, session_id=sid, order_id=key[1], trade_id=key[0],
                          side=side, quantity=fill["size"], price=fill["price"],
                          match_time=stamp(actual), match_epoch=actual,
                          confirmed_observed_time=stamp(t), confirmed_epoch=t,
                          matched_observed_time=stamp(observations[key].get("MATCHED")),
                          matched_observed_epoch=observations[key].get("MATCHED"),
                          confirmation_lag_s=t-actual if actual else None, source_line=line)
            fills_out.append(record)
            order_fills[key[1]].append(record)
        merges = {q["merge_id"]: (line, t, q) for line, t, q in by["sm_merge_result"]
                  if q.get("status") == "CONFIRMED"}
        merged = sum((D(q["amount_raw"])/1000000 for _, _, q in merges.values()), Decimal(0))
        released = sum((D(q["released_cost_usdc"]) for _, _, q in merges.values()), Decimal(0))
        pair_pnl = merged-released
        residual_cost = sum(costs.values())-released
        cash_flow = merged-sum(costs.values())
        assert len(confirmed) == result["confirmed_fills"], (label, "fill count")
        assert abs(pair_pnl-D(result["realized_merge_pnl"])) < D(".000001"), (label, "PnL")
        for side in ("YES", "NO"):
            assert abs(quantities[side]-merged-D(result[f"confirmed_{side.lower()}"])) < D(".000001"), (label, side)
        local_windows = []
        for wid, (line, intent, sizing) in sorted(intents.items(), key=lambda a:a[1][1]["post_started"] or a[1][1]["frozen_at"]):
            placement, outcome = placements[wid], results[wid]
            order = placement.get("order_id") or placement["expected_order_id"]
            final_placement = outcome["placement_results"][0]
            final_outcomes[final_placement["outcome"]] += 1
            close_line, close = closes[wid]
            signal = intent.get("final_signal") or intent.get("signal") or {}
            quote = intent["quotes"][0]
            fills = order_fills[order]
            submitted = intent["post_started"] is not None
            began = intent["post_started"] or intent["frozen_at"]
            row = dict(market=label, session_id=sid, window=wid.rsplit(":", 1)[-1], phase=intent["phase"],
                       signal=signal.get("label"), obi=signal.get("imbalance"),
                       submitted=submitted, aborted_reason=intent.get("aborted_reason"),
                       post_time=stamp(began+offset) if submitted else "未发送", post_epoch=began+offset,
                       post_ms=(intent["post_returned"]-began)*1000 if submitted else None,
                       side=quote["outcome"], price=quote["price"], size=quote["size"],
                       outcome_initial=placement["outcome"], outcome_final=final_placement["outcome"],
                       close_time=stamp(close["requested_monotonic"]+offset), close_reason=close["reason"],
                       close_signal=(close.get("signal") or {}).get("imbalance"),
                       post_to_close_s=close["requested_monotonic"]-began if submitted else None,
                       original_deadline=stamp(intent["deadline"]+offset if intent["deadline"] else None), close_barrier=outcome["close_barrier"],
                       block_reason=outcome.get("block_reason"),
                       final_close_time=stamp(outcome["end_time"]),
                       post_to_safe_s=outcome["end_time"]-(began+offset) if submitted else None,
                       close_to_safe_s=outcome["end_time"]-(close["requested_monotonic"]+offset),
                       fill_quantity=sum(f["quantity"] for f in fills),
                       match_times=";".join(f["match_time"] if f["match_epoch"] else "本地MATCHED " + f["matched_observed_time"] for f in fills),
                       fill_count=len(fills), initial_p95_ms=(intent.get("health") or {}).get("p95_ms"),
                       target_delta=sizing.get("configured_target_delta"), ideal_repair=sizing.get("ideal_repair_size"),
                       repair_fallback=";".join(sizing.get("repair_sizing_fallback", [])),
                       source_line=line, close_source_line=close_line, order_id=order,
                       raw_error=";".join(z.get("errorMsg", "") for z in (placement.get("raw_response") or []) if isinstance(z, dict)))
            windows.append(row)
            local_windows.append(row)
        for line, t, q in by["sm_network_request"]:
            requests.append(dict(market=label, **q))
        for line, t, q in by["sm_obi_wait"]:
            waits_all[q["reason"]] += 1
            early_signals.append(dict(market=label, time=stamp(t), epoch=t, reason=q["reason"],
                                      signal=(q.get("signal") or {}).get("label"),
                                      obi=(q.get("signal") or {}).get("imbalance"), line=line))
        for line, t, q in by["sm_obi_signal_changes"]:
            active = closes[q["window_id"]][1]["requested_monotonic"]
            signal_histories.append(dict(market=label, window=q["window_id"].rsplit(":",1)[-1],
                                        count=len(q["changes"]), active_count=sum(z[1]<=active for z in q["changes"]),
                                        overwritten=q["history_overwritten"]))
        for line, t, q in by["sm_network_transition"]:
            event_time = q.get("event_monotonic", t-offset)+offset
            paused.append(dict(market=label, time=stamp(event_time), epoch=event_time, line=line, **q))
        relevant = ("sm_network_transition", "sm_obi_wait", "sm_trade", "sm_merge_result", "sm_run_result")
        for line, x, q in rows:
            if x["kind"] in relevant and (x["kind"]!="sm_trade" or q["status"]=="CONFIRMED"):
                event_epoch = q.get("event_monotonic", x["ts"]-offset)+offset if x["kind"]=="sm_network_transition" else x["ts"]
                events.append(dict(market=label, epoch=event_epoch, time=stamp(event_epoch), kind=x["kind"], line=line,
                                   side=token_side.get(q.get("token_id")), data=q))
        first = next((w for w in local_windows if w['submitted']), {})
        waits = Counter(q["reason"] for _, _, q in by["sm_obi_wait"])
        close_count = Counter(w["close_reason"] for w in local_windows)
        market = dict(market=label, epoch=epoch, session_id=sid, slug=session["slug"],
                      execution_status=result["execution_status"], final_barrier=result["final_close_barrier"],
                      windows=len(local_windows), submitted=sum(w['submitted'] for w in local_windows),
                      normal=sum(w["phase"]=="NORMAL" and w['submitted'] for w in local_windows),
                      rebalance=sum(w["phase"]=="REBALANCE" and w['submitted'] for w in local_windows),
                      first_side=first.get("side"), first_obi=first.get("obi"), first_post=first.get("post_time"),
                      first_observed=stamp(head["ts"]), start_delay_s=head["ts"]-epoch,
                      filled_orders=sum(w["fill_quantity"]>0 for w in local_windows), fill_records=len(confirmed),
                      bought_yes=float(quantities["YES"]), bought_no=float(quantities["NO"]),
                      merged_pairs=float(merged), pair_pnl=float(pair_pnl), residual_cost=float(residual_cost),
                      residual_yes=result["confirmed_yes"], residual_no=result["confirmed_no"],
                      scoped_cash_flow=float(cash_flow), starting_balance=session["starting_official_balance_usdc"],
                      end_time=stamp(stopped), end_epoch=stopped,
                      reverse_closes=close_count["SIGNAL_REVERSED"], deadline_closes=close_count["ORDER_DEADLINE"],
                      stale_closes=close_count["WS_BOOK_STALE"],
                      no_legal_waits=waits["NO_LEGAL_SIGNAL_CANDIDATE"],
                      network_pauses=sum(q["state"]=="PAUSED" for _, _, q in by["sm_network_transition"]),
                      max_writer_queue=max(x["data"].get("writer_queue_depth",0) for _,x,_ in rows),
                      dropped_diagnostics=max(x["data"].get("diagnostics_dropped",0) for _,x,_ in rows),
                      source=source, result_line=result_line)
        markets.append(market)
    markets.sort(key=lambda r:r["epoch"])
    windows.sort(key=lambda r:r["post_epoch"])
    assert len(markets)==12 and [r["epoch"] for r in markets]==list(range(START,END,300))
    sent = [w for w in windows if w['submitted']]
    totals = dict(markets=len(markets), windows=len(windows), submitted=len(sent),
                  executions=dict(Counter(m["execution_status"] for m in markets)),
                  phases=dict(Counter(w["phase"] for w in sent)), outcomes=dict(final_outcomes),
                  close_reasons=dict(Counter(w["close_reason"] for w in windows)),
                  phase_close_reasons={p:dict(Counter(w["close_reason"] for w in windows if w["phase"]==p)) for p in ("NORMAL","REBALANCE")},
                  fill_records=len(fills_out), filled_orders=sum(w["fill_quantity"]>0 for w in windows),
                  traded_markets=sum(m["fill_records"]>0 for m in markets),
                  residual_markets=sum(m["residual_yes"]+m["residual_no"]>0 for m in markets),
                  pair_pnl=sum(m["pair_pnl"] for m in markets), residual_cost=sum(m["residual_cost"] for m in markets),
                  scoped_cash_flow=sum(m["scoped_cash_flow"] for m in markets),
                  merged_pairs=sum(m["merged_pairs"] for m in markets),
                  waits=dict(waits_all), pause_reasons=dict(Counter(p["reason"] for p in paused if p["state"]=="PAUSED")),
                  request_outcomes=dict(Counter(r["outcome"] for r in requests)),
                  request_methods=dict(Counter(r["method"] for r in requests)),
                  post_ms={"median":statistics.median(w["post_ms"] for w in sent), "p95":percentile([w["post_ms"] for w in sent],.95), "max":max(w["post_ms"] for w in sent)},
                  phase_fill_rates={p:dict(submitted=sum(w["phase"]==p for w in sent),filled=sum(w["phase"]==p and w["fill_quantity"]>0 for w in sent)) for p in ("NORMAL","REBALANCE")},
                  malformed_lines=errors, history_overwritten=sum(r["overwritten"] for r in signal_histories))
    write_csv("markets.csv", markets)
    write_csv("windows.csv", windows)
    write_csv("fills.csv", fills_out)
    write_csv("signal_waits.csv", sorted(early_signals,key=lambda r:r["epoch"]))
    (OUT/"analysis.json").write_text(json.dumps(dict(totals=totals,markets=markets,windows=windows,
        fills=fills_out,network_transitions=paused,signal_histories=signal_histories,
        signal_waits=early_signals,provenance=provenance),ensure_ascii=False,indent=2))
    # Full per-market timeline, including all orders and material lifecycle events.
    lines=["# 12 场逐单时间线（北京时间，2026-10-17 22:45–23:45）", "",
           "范围冻结为 12 个连续已结束市场；未过滤异常结果。POST 时间按 monotonic/墙钟对齐。",
           "本批全部 33 条成交缺少可用官方撮合时间（缓存 timestamp=0）；成交列使用本地首次 MATCHED 观察时间，相关事件使用 CONFIRMED 观察时间，均不能称为官方撮合时刻。NONE/INVALID/等待日志不构成完整逐帧信号流。", ""]
    for m in markets:
        lines += [f"## {m['market']} / {m['session_id']}", "",
                  f"[{m['slug']}]({m['source']}:1)；结束 {m['end_time']}，{m['execution_status']} / {m['final_barrier']}。",
                  f"合并毛收益 {m['pair_pnl']:+.6f} USDC；期末 YES {m['residual_yes']} / NO {m['residual_no']}；残仓成本 {m['residual_cost']:.6f}。", "",
                  "|轮次|阶段|POST|信号/OBI|BUY 价格×数量|POST ms|关闭请求/原因/反向OBI|终态/核验结束|确认成交|",
                  "|---|---|---|---|---|---:|---|---|---|"]
        for w in windows:
            if w['session_id']!=m['session_id']:continue
            signal=f"{w['signal'] or '库存补仓'} / {float(w['obi']):+.3f}" if w['obi'] is not None else '库存补仓'
            post_ms=f"{w['post_ms']:.0f}" if w['submitted'] else '—'
            reversal=f" / {float(w['close_signal']):+.3f}" if w['close_signal'] is not None else ''
            lines.append(f"|[{w['window']}]({m['source']}:{w['source_line']})|{w['phase']}|{w['post_time']}|{signal}|{w['side']} {w['price']}×{w['size']}|{post_ms}|{w['close_time']} {w['close_reason']}{reversal}|{w['outcome_final']} / {w['close_barrier']} {w['block_reason'] or ''} / {w['final_close_time']}|{w['fill_quantity']}；{w['match_times']}|")
        lines += ["", "相关事件：", ""]
        for event in sorted((e for e in events if e['market']==m['market']),key=lambda e:e['epoch']):
            q=event['data'];k=event['kind']
            if k=='sm_network_transition':desc=f"网络 {q['previous']} → {q['state']}：{q['reason']}"
            elif k=='sm_obi_wait':desc=f"等待 {q['reason']}；{(q.get('signal') or {}).get('label')} / {(q.get('signal') or {}).get('imbalance')}"
            elif k=='sm_trade':desc=f"本地观察 CONFIRMED：{event['side']} {q['size']} @ {q['price']}；trade={q['trade_id']}"
            elif k=='sm_merge_result':desc=f"合并 {D(q['amount_raw'])/1000000} 对，状态 {q['status']}，释放成本 {q.get('released_cost_usdc')}"
            else:continue
            lines.append(f"- [{event['time']}]({m['source']}:{event['line']}) {desc}")
        lines.append("")
    (OUT/"timeline.md").write_text("\n".join(lines))
    print(json.dumps(totals,ensure_ascii=False,indent=2))
    for m in markets:
        print(m['market'],m['normal'],m['rebalance'],m['first_side'],m['first_obi'],m['filled_orders'],m['pair_pnl'],m['residual_yes'],m['residual_no'])
    return totals


if __name__=="__main__":
    analyze()


{
  "markets": 12,
  "windows": 213,
  "submitted": 209,
  "executions": {
    "COMPLETED": 12
  },
  "phases": {
    "NORMAL": 129,
    "REBALANCE": 80
  },
  "outcomes": {
    "ACK": 190,
    "UNKNOWN": 4,
    "REJECTED": 15,
    "ABORTED": 4
  },
  "close_reasons": {
    "ORDER_DEADLINE": 68,
    "SIGNAL_REVERSED": 77,
    "PLACEMENT_NOT_ACKNOWLEDGED": 9,
    "WS_BOOK_STALE": 55,
    "MARKET_SUBMISSION_CUTOFF": 3,
    "WS_RECONNECT_REQUIRED": 1
  },
  "phase_close_reasons": {
    "NORMAL": {
      "ORDER_DEADLINE": 48,
      "SIGNAL_REVERSED": 77,
      "MARKET_SUBMISSION_CUTOFF": 3,
      "PLACEMENT_NOT_ACKNOWLEDGED": 3,
      "WS_RECONNECT_REQUIRED": 1
    },
    "REBALANCE": {
      "PLACEMENT_NOT_ACKNOWLEDGED": 6,
      "WS_BOOK_STALE": 55,
      "ORDER_DEADLINE": 20
    }
  },
  "fill_records": 33,
  "filled_orders": 27,
  "traded_markets": 10,
  "residual_markets": 1,
  "pair_pnl": -7.197609561752988,
  "residual_cost": 0.012590438247011951,
  "scoped_cash_flow": -7.2102,
  "m

In [2]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/polymaker').is_dir())
import os, sys
os.chdir(ROOT)
REVIEW = ROOT / 'docs/development/reviews/20261017_2245_2345'
sys.path.insert(0, str(REVIEW))
"""Read-only reconciliation and cache evidence; no trading imports or network."""
import hashlib
import json
import statistics
from collections import Counter, defaultdict
from decimal import Decimal
from pathlib import Path

from analyze import OUT, percentile, stamp, write_csv


def quant(values):
    return dict(n=len(values), median=statistics.median(values),
                p95=percentile(values, .95), max=max(values)) if values else dict(n=0)


def main():
    data = json.loads((OUT / 'analysis.json').read_text())
    windows = data['windows']
    sent = [w for w in windows if w['submitted']]
    merges, barriers, gaps, observer, source_hashes, changes = [], [], [], [], [], []
    counts = Counter()
    requests = []
    issues = []
    latencies = defaultdict(list)
    cache_invariants = []
    previous = None
    for m in data['markets']:
        rows = [(i, json.loads(l)) for i, l in enumerate(Path(m['source']).read_text().splitlines(), 1)]
        ownfills = [f for f in data['fills'] if f['session_id'] == m['session_id']]
        network=[r['data']['payload'] for _,r in rows if r['kind']=='sm_network_request']
        offset=statistics.median(q['completed_wall']-q['completed_monotonic'] for q in network)
        closes={r['data']['payload']['window_id']:r['data']['payload']['requested_monotonic']
                for _,r in rows if r['kind']=='sm_obi_close_requested'}
        for line, r in rows:
            q = r['data']['payload']
            kind = r['kind']
            if kind == 'sm_obi_signal_changes':
                for change in q['changes']:
                    changes.append(dict(market=m['market'],window=q['window_id'].rsplit(':',1)[-1],
                        time=stamp(change[1]+offset),epoch=change[1]+offset,signal=change[3],
                        before_close=change[1]<=closes[q['window_id']],source_line=line))
            if kind == 'sm_merge_result' and q['status'] == 'CONFIRMED':
                merges.append(dict(market=m['market'], time=stamp(r['ts']), epoch=r['ts'],
                    pairs=int(q['amount_raw'])/1e6, released_cost=q['released_cost_usdc'],
                    balance_before=int(q['collateral_before_raw'])/1e6,
                    balance_after=int(q['collateral_after_raw'])/1e6,
                    source=m['source'], source_line=line))
            if kind == 'sm_close_barrier':
                reason=q.get('reason') or ''
                category = reason.rsplit(':', 1)[-1]
                counts['barrier:' + q['result'] + ':' + category] += 1
                barriers.append(dict(market=m['market'], time=stamp(r['ts']), epoch=r['ts'],
                    result=q['result'], reason=reason, source_line=line))
            if kind == 'sm_network_request':
                ep=q['endpoint']
                if ep.startswith('/data/order/'): ep='/data/order/:id'
                latencies[q['method']+':'+ep].append(q['duration_ms'])
            if kind == 'sm_sync_cache_read':
                boundary=q['operation_after_monotonic']
                required={'time', 'open_orders', 'trades'}
                if q['purpose']=='account': required|={'capital', 'positions'}
                valid=all(q['sources'][name]['started_monotonic'] >= boundary for name in required)
                counts['cache_scopes'] += 1
                if not valid: cache_invariants.append([m['market'], line])
        if previous:
            gap=Decimal(str(m['starting_balance']))-Decimal(str(previous['starting_balance']))-Decimal(str(previous['scoped_cash_flow']))
            gaps.append(dict(previous=previous['market'], next=m['market'], unexplained=float(gap)))
        previous=m
        root=Path('research_outputs/periodic_sync') / m['session_id']
        manifest=json.loads((root/'manifest.json').read_text())
        summary=json.loads((root/'summary.json').read_text())
        source_hashes.append(manifest['source_sha256'])
        for name in ['manifest.json', 'summary.json', 'events.jsonl', 'snapshots.jsonl', 'strategy_refs.jsonl']:
            f=root/name
            h=hashlib.sha256()
            with f.open('rb') as stream:
                while chunk:=stream.read(1024*1024):h.update(chunk)
            data['provenance'].append(dict(path=str(f.resolve()), bytes=f.stat().st_size, sha256=h.hexdigest()))
        assert manifest['session_id']==m['session_id']
        assert manifest['market']['slug']==m['slug']
        evcounts=Counter()
        for line,l in enumerate((root/'events.jsonl').open(),1):
            e=json.loads(l)
            if e['kind']=='sync_http':
                evcounts['http']+=1
                evcounts['http:'+str(e.get('http_status'))+':'+str(e.get('error_type'))]+=1
                requests.append(dict(market=m['market'], line=line, **e))
            if e['kind']=='sync_round' and not e.get('shutdown'):
                evcounts['round:'+e['lane']]+=1
                evcounts['incomplete:'+e['lane']]+=not e['complete']
                evcounts['overrun:'+e['lane']]+=e['overrun']
            if e['kind']=='sync_consistency':
                evcounts['consistency']+=1
                if not e['observations_agree']:
                    issues.append(dict(market=m['market'], line=line, **e))
        # Verify persisted snapshot versions, market scope, and local fills.
        persisted_confirmed={}
        max_timestamp=0
        snapshot_issues=[]
        for line,l in enumerate((root/'snapshots.jsonl').open(),1):
            s=json.loads(l)
            if s['version']!=line or s['session_id']!=m['session_id']:
                snapshot_issues.append([line,'identity_or_version'])
            if s.get('market') and s['market']!=manifest['market']:
                snapshot_issues.append([line,'market'])
            if s.get('trading_authorized') is not False:
                snapshot_issues.append([line,'authority'])
            trades=s.get('last_good',{}).get('trades',{}).get('data') or []
            for t in trades:
                for leg in t['maker_legs']:
                    key=(t['trade_id'],leg['order_id'])
                    if t['status']=='CONFIRMED' and any(f['trade_id']==key[0] and f['order_id']==key[1] for f in ownfills):
                        persisted_confirmed[key]=leg
                        max_timestamp=max(max_timestamp,t['timestamp'])
        assert not snapshot_issues, snapshot_issues
        assert {(f['trade_id'],f['order_id']) for f in ownfills}==set(persisted_confirmed)
        assert all(Decimal(str(persisted_confirmed[(f['trade_id'],f['order_id'])]['matched_amount']))==Decimal(str(f['quantity']))
                   and Decimal(str(persisted_confirmed[(f['trade_id'],f['order_id'])]['price']))==Decimal(str(f['price'])) for f in ownfills)
        observer.append(dict(market=m['market'], session_id=m['session_id'],
            execution_reads_from_cache=manifest['execution_reads_from_cache'],
            source_sha256=manifest['source_sha256'], snapshots=summary['snapshot_count'],
            status=summary.get('status'), observer_status=summary['observer_status'],
            dropped_events=summary['dropped_events'], dropped_references=summary['dropped_references'],
            reference_market_or_history_gaps=summary['reference_market_or_history_gaps'],
            io_drained=summary['io_drained'], failure=summary['failure'],
            confirmed_snapshot_keys=len(persisted_confirmed), max_trade_timestamp=max_timestamp,
            counters=dict(evcounts)))
    unknown=[w for w in sent if w['outcome_final']=='UNKNOWN']
    canceled=[w for w in sent if 'Already canceled' in w['raw_error']]
    rejected=[w for w in sent if w['outcome_final']=='REJECTED']
    last_merge=merges[-1]
    balance_end=last_merge['balance_after']
    balance_change=balance_end-data['markets'][0]['starting_balance']
    evidence=dict(scope='2026-10-17 22:45–23:45 Asia/Shanghai; latest 12 logged BTC five-minute markets',
        sent_outcomes=dict(Counter(w['outcome_final'] for w in sent)),
        canceled_count=len(canceled), canceled_outcomes=dict(Counter(w['outcome_final'] for w in canceled)),
        post_only_cross_count=sum('crosses book' in w['raw_error'] for w in rejected),
        unknown_windows=[{k:w[k] for k in ['market','window','post_time','phase','outcome_final','close_barrier','block_reason','final_close_time']} for w in unknown],
        blocked_windows=[{k:w[k] for k in ['market','window','post_time','outcome_final','block_reason','final_close_time']} for w in windows if w['close_barrier']=='BLOCKED'],
        phase_metrics={phase:dict(
            submitted=len(ws), outcomes=dict(Counter(w['outcome_final'] for w in ws)),
            filled_orders=sum(w['fill_quantity']>0 for w in ws),
            close_reasons=dict(Counter(w['close_reason'] for w in ws)),
            post_to_close_s=quant([w['post_to_close_s'] for w in ws]),
            close_to_window_end_s=quant([w['close_to_safe_s'] for w in ws]),
            unfilled_ack_post_to_close_s=quant([w['post_to_close_s'] for w in ws if w['outcome_final']=='ACK' and not w['fill_quantity']]),
            ack_stale=sum(w['close_reason']=='WS_BOOK_STALE' and w['outcome_final']=='ACK' for w in ws))
            for phase in ['NORMAL','REBALANCE'] for ws in [[w for w in sent if w['phase']==phase]]},
        starting_balance=data['markets'][0]['starting_balance'], final_observed_balance=balance_end,
        final_balance_time=last_merge['time'], observed_balance_change=balance_change,
        scoped_cash_flow=data['totals']['scoped_cash_flow'],
        unexplained_cash_flow=balance_change-data['totals']['scoped_cash_flow'],
        between_market_gaps=gaps, merges=merges, observer=observer,
        same_sync_source_hashes=all(h==source_hashes[0] for h in source_hashes),
        current_source_matches={name:hashlib.sha256((Path('src/polymaker')/name).read_bytes()).hexdigest()==digest for name,digest in source_hashes[0].items()},
        counts=dict(counts), cache_source_boundary_failures=cache_invariants,
        consistency_disagreements=issues,
        request_metrics={k:quant(v) for k,v in latencies.items()},
        http_observer_counts=dict(sum((Counter(o['counters']) for o in observer),Counter())),
        non_200_observer_requests=[r for r in requests if r.get('http_status')!=200],
        matched_to_confirmed_observation_s=quant([f['confirmed_epoch']-f['matched_observed_epoch'] for f in data['fills'] if f['matched_observed_epoch']]),
        max_writer_queue=max(m['max_writer_queue'] for m in data['markets']),
        diagnostics_dropped=max(m['dropped_diagnostics'] for m in data['markets']),
        provenance=data['provenance'])
    write_csv('merges.csv',merges)
    write_csv('close_barriers.csv',barriers)
    write_csv('balance_gaps.csv',gaps)
    write_csv('signal_changes.csv',sorted(changes,key=lambda r:r['epoch']))
    (OUT/'evidence.json').write_text(json.dumps(evidence,ensure_ascii=False,indent=2)+'\n')
    print(json.dumps({k:v for k,v in evidence.items() if k not in ['observer','non_200_observer_requests','consistency_disagreements','provenance','merges','request_metrics']},ensure_ascii=False,indent=2))
    print('NON200',len(evidence['non_200_observer_requests']), Counter((r.get('http_status'),r.get('error_type')) for r in evidence['non_200_observer_requests']))
    print('CONSISTENCY',len(issues))
    print('OBSERVER',[(o['market'],o['status'],o['snapshots'],o['counters']) for o in observer])
    return evidence


if __name__=='__main__':main()


{
  "scope": "2026-10-17 22:45–23:45 Asia/Shanghai; latest 12 logged BTC five-minute markets",
  "sent_outcomes": {
    "ACK": 190,
    "REJECTED": 15,
    "UNKNOWN": 4
  },
  "canceled_count": 11,
  "canceled_outcomes": {
    "REJECTED": 7,
    "UNKNOWN": 4
  },
  "post_only_cross_count": 8,
  "unknown_windows": [
    {
      "market": "23:30",
      "window": "w3",
      "post_time": "23:30:36.211",
      "phase": "REBALANCE",
      "outcome_final": "UNKNOWN",
      "close_barrier": "BLOCKED",
      "block_reason": "SYNC_CACHE_NOT_READY",
      "final_close_time": "23:30:47.912"
    },
    {
      "market": "23:35",
      "window": "w8",
      "post_time": "23:36:13.023",
      "phase": "NORMAL",
      "outcome_final": "UNKNOWN",
      "close_barrier": "BLOCKED",
      "block_reason": "SYNC_CACHE_NOT_READY",
      "final_close_time": "23:36:24.198"
    },
    {
      "market": "23:40",
      "window": "w11",
      "post_time": "23:41:15.346",
      "phase": "REBALANCE",
      "outcom

In [3]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/polymaker').is_dir())
import os
os.chdir(ROOT)
REVIEW = ROOT / 'docs/development/reviews/20261017_2245_2345'
"""Measure cache cadence, source age and real shared GET usage from frozen logs."""
import bisect
import json
import math
import statistics
from collections import Counter, defaultdict
from pathlib import Path

OUT=REVIEW


def quant(values):
    v=sorted(values)
    return dict(n=len(v),p50=statistics.median(v),p95=v[math.ceil(len(v)*.95)-1],max=v[-1]) if v else dict(n=0)


def main():
    data=json.loads((OUT/'analysis.json').read_text())
    values=defaultdict(list)
    counter=Counter()
    permarket=[]
    for m in data['markets']:
        root=Path('research_outputs/periodic_sync')/m['session_id']
        events=[json.loads(l) for l in (root/'events.jsonl').open()]
        readids={e['read_id'] for e in events if e['kind']=='sync_http' and e['read_id']!='bootstrap'}
        sourcehttp=Counter(e['endpoint'] for e in events if e['kind']=='sync_http')
        lanes=defaultdict(list)
        for e in events:
            if e['kind']=='sync_round' and not e.get('shutdown'):
                lane=e['lane'];lanes[lane].append(e['started_monotonic'])
                values['round_ms:'+lane].append(e['duration_s']*1000)
                counter['rounds:'+lane]+=1
                counter['incomplete:'+lane]+=not e['complete']
                counter['overruns:'+lane]+=e['overrun']
        for lane,ts in lanes.items():values['interval_ms:'+lane].extend((b-a)*1000 for a,b in zip(ts,ts[1:]))
        # Fixed 100 ms grid avoids overweighting frames published by a faster lane.
        snaps=[json.loads(l) for l in (root/'snapshots.jsonl').open()]
        pub=[s['published_monotonic'] for s in snaps]
        names=['open_orders','trades','capital','positions']
        valid=[s for s in snaps if s['market'] and all(n in s['last_good'] for n in names)]
        if valid:
            t=valid[0]['published_monotonic']
            while t<pub[-1]:
                snap=snaps[bisect.bisect_right(pub,t)-1]
                for name in names:
                    values['timegrid_age_ms:'+name].append((t-snap['last_good'][name]['started_monotonic'])*1000)
                t+=.1
        mainhttp=Counter();native=Counter();shared=Counter()
        for l in Path(m['source']).open():
            r=json.loads(l);q=r['data']['payload']
            if r['kind']=='sm_network_request' and q['method']=='GET':
                mainhttp[q['endpoint']]+=1
                if q['operation_id'] in readids:shared[q['endpoint']]+=1
                else:native[q['endpoint']]+=1
            if r['kind']=='sm_sync_cache_read':
                for n,s in q['sources'].items():
                    values['consumed_age_ms:'+q['purpose']+':'+n].append((r['data']['occurred_monotonic']-s['started_monotonic'])*1000)
        counter['main_gets']+=sum(mainhttp.values())
        counter['main_gets_shared']+=sum(shared.values())
        counter['main_gets_native']+=sum(native.values())
        counter['collector_gets']+=sum(sourcehttp.values())
        for ep,n in sourcehttp.items():counter['collector:'+('/data/order/:id' if ep.startswith('/data/order/') else ep)]+=n
        for ep,n in native.items():counter['native:'+('/data/order/:id' if ep.startswith('/data/order/') else ep)]+=n
        permarket.append(dict(market=m['market'],main_gets=sum(mainhttp.values()),shared_gets=sum(shared.values()),native_gets=sum(native.values())))
    old=json.loads((OUT.parent/'20261017_sync_attached_12/evidence.json').read_text())
    previous_main=sum(n for k,n in old['strategy_request_counts']['attached'].items() if k.startswith('GET:'))
    previous_observer=old['counter']['observer_method:GET']
    current_union=counter['collector_gets']+counter['main_gets_native']
    result=dict(scope='Frozen 12 markets, 2026-10-17 22:45–23:45 Asia/Shanghai',
        counters=dict(counter),metrics={k:quant(v) for k,v in values.items()},per_market=permarket,
        comparison=dict(previous_scope='2026-10-17 20:20–21:20, observer only',
            previous_main_timed_gets=previous_main,previous_collector_gets=previous_observer,
            previous_union_covered_gets=previous_main+previous_observer,
            current_union_covered_gets=current_union,
            covered_get_reduction_fraction=1-current_union/(previous_main+previous_observer),
            caveat='Descriptive different market cohorts, not A/B. Main timing excludes some startup, settlement and positions calls. Shared collector requests deduplicated by read_id/operation_id.'),
        source_files=['analysis.json','evidence.json','../20261017_sync_attached_12/evidence.json',
            'each selected session events.jsonl, snapshots.jsonl and strategy journal'])
    (OUT/'cache_performance.json').write_text(json.dumps(result,indent=2,ensure_ascii=False)+'\n')
    print(json.dumps(result,indent=2,ensure_ascii=False))


if __name__=='__main__':main()


{
  "scope": "Frozen 12 markets, 2026-10-17 22:45–23:45 Asia/Shanghai",
  "counters": {
    "rounds:orders": 3620,
    "incomplete:orders": 2,
    "overruns:orders": 7,
    "rounds:account": 1789,
    "incomplete:account": 0,
    "overruns:account": 0,
    "main_gets": 14177,
    "main_gets_shared": 12628,
    "main_gets_native": 1549,
    "collector_gets": 15246,
    "collector:/auth/derive-api-key": 12,
    "collector:/time": 3634,
    "collector:/balance-allowance": 1790,
    "collector:/data/orders": 3622,
    "collector:/data/trades": 3622,
    "collector:/v2/positions": 1790,
    "collector:/data/order/:id": 776,
    "native:/tick-size": 1402,
    "native:/data/orders": 27,
    "native:/version": 39,
    "native:/balance-allowance": 27,
    "native:/data/trades": 27,
    "native:/time": 27
  },
  "metrics": {
    "round_ms:orders": {
      "n": 3620,
      "p50": 211.99168800376356,
      "p95": 428.56333684176207,
      "max": 2853.3258312381804
    },
    "round_ms:account": {


In [4]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/polymaker').is_dir())
import os
os.chdir(ROOT)
REVIEW = ROOT / 'docs/development/reviews/20261017_2245_2345'
"""Replay 23:25 capital admission arithmetic from persisted evidence only."""
import csv
import json
from decimal import Decimal, ROUND_CEILING
from pathlib import Path

OUT = REVIEW
D = lambda v: Decimal(str(v))


def main():
    data = json.loads((OUT / 'analysis.json').read_text())
    market = next(m for m in data['markets'] if m['market'] == '23:25')
    rows = [(i, json.loads(l)) for i, l in enumerate(Path(market['source']).open(), 1)]
    decisions = {r['data']['payload']['intent']['window_id']: (i, r['data']['payload'])
                 for i, r in rows if r['kind'] == 'sm_obi_final_decision'}
    windows = [w for w in data['windows'] if w['session_id'] == market['session_id']]
    results = {r['data']['payload']['window_id']: r['data']['payload']
               for _, r in rows if r['kind'] == 'sm_window_result'}
    cap = D(15)
    previous = D(0)
    audit = []
    # Before the first merge, confirmed cost remains reserved after pair balance.
    for w in windows:
        if w['window'] == 'w8':
            break
        line, q = decisions[market['session_id'] + ':' + w['window']]
        intent = q['intent']
        quote = intent['quotes'][0]
        cost = D(quote['price']) * D(quote['size'])
        snapshot = intent.get('final_snapshot') or intent['snapshot']
        opposite = 'no' if quote['outcome'] == 'YES' else 'yes'
        opposite_price = D(snapshot[opposite]['best_bid'])
        # All NORMAL entries in this case start at delta=0 and buy 5 shares.
        repair_size = max(D(5), (D(1)/opposite_price*100).to_integral_value(rounding=ROUND_CEILING)/100)
        opposite_cost = repair_size * opposite_price
        record = dict(window=w['window'], phase=w['phase'], post_time=w['post_time'],
            side=quote['outcome'], price=str(quote['price']), shares=str(quote['size']),
            committed_before=str(previous), entry_commitment=str(cost),
            hypothetical_committed_after_entry=str(previous+cost),
            hypothetical_cap_headroom_after_entry=str(cap-previous-cost),
            existing_cap_check_passes=previous+cost<=cap,
            actual_outcome=w['outcome_final'], actual_filled_shares=w['fill_quantity'],
            actual_committed_at_window_end=str(results[w['session_id']+':'+w['window']]['capital_committed']),
            source_line=line, source=market['source'])
        if w['phase'] == 'NORMAL':
            record.update(opposite_bid=str(opposite_price), repair_shares_at_current_bid=str(repair_size),
                repair_cost_at_current_bid=str(opposite_cost),
                hypothetical_full_cycle_committed=str(previous+cost+opposite_cost),
                passes_current_bid_repair_budget=previous+cost+opposite_cost<=cap,
                cannot_afford_even_one_dollar_repair=cap-previous-cost<D(1),
                repair_reserve_at_tick_max=str(D(quote['size'])*(1-D(snapshot[opposite]['tick']))))
        audit.append(record)
        previous=D(results[w['session_id']+':'+w['window']]['capital_committed'])
    normals=[r for r in audit if r['phase']=='NORMAL']
    assert all(r['existing_cap_check_passes'] for r in audit)
    assert [r['window'] for r in normals if r['cannot_afford_even_one_dollar_repair']]==['w5','w6','w7']
    assert [r['window'] for r in normals if not r['passes_current_bid_repair_budget']]==['w5','w6','w7']
    assert D(audit[-1]['actual_committed_at_window_end'])==D('14.7')
    result=dict(scope='2026-10-17 23:25 BTC five-minute market; local POST times Asia/Shanghai',
        starting_wallet_balance=str(market['starting_balance']), session_cap='15',
        actual_cycle_entry_that_created_stranded_gap='w7', first_counterfactual_rejection='w5',
        assumptions=['NORMAL entries in this sample start at zero delta and request 5 shares.',
            'Full fill is a capital exposure scenario, not a prediction of fill.',
            'Opposite current bid sizes/costs illustrate immediate budget feasibility only; they do not guarantee future execution or price.',
            'Minimum legal BUY notional is 1 USDC; all cases require an additional opposite BUY after a full NORMAL fill.',
            'Existing committed amounts are journal window results, independently reconciled against confirmed fills in analysis.json.'],
        windows=audit)
    (OUT/'capital_15.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
    fields=list(dict.fromkeys(k for r in audit for k in r))
    with (OUT/'capital_15.csv').open('w') as f:
        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader();writer.writerows(audit)
    print(json.dumps(result,ensure_ascii=False,indent=2))


if __name__=='__main__':main()


{
  "scope": "2026-10-17 23:25 BTC five-minute market; local POST times Asia/Shanghai",
  "starting_wallet_balance": "22.382732",
  "session_cap": "15",
  "actual_cycle_entry_that_created_stranded_gap": "w7",
  "first_counterfactual_rejection": "w5",
  "assumptions": [
    "NORMAL entries in this sample start at zero delta and request 5 shares.",
    "Full fill is a capital exposure scenario, not a prediction of fill.",
    "Opposite current bid sizes/costs illustrate immediate budget feasibility only; they do not guarantee future execution or price.",
    "Minimum legal BUY notional is 1 USDC; all cases require an additional opposite BUY after a full NORMAL fill.",
    "Existing committed amounts are journal window results, independently reconciled against confirmed fills in analysis.json."
  ],
  "windows": [
    {
      "window": "w1",
      "phase": "NORMAL",
      "post_time": "23:25:22.864",
      "side": "YES",
      "price": "0.52",
      "shares": "5.0",
      "committed_befor